# Linear Regression - Lasso tuning

### __Introduction__

- We will start by using this model mostly as a baseline for our other models.
- A priori, we think that this will most likely not be our best performing model, mostly because it assumes linearity of features, which may not be the case. 

## 1. 

In [1]:
# 1) Imports 
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.model_selection import ParameterSampler

# 2) Import pre processing helpers
#    -> this should define: full_train_dataset, cat_feat, num_feat,
#       basic_string_transformer, def_string_basic_transformer,
#       preprocess_categorical, MyTargetEncoder, MyOneHotEncoder, etc.
%run 05_0_preproc_helpers.ipynb  

# 3) Define target
TARGET_COL = "price"

# 4) Separate X and y from the treated dataset
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# Numerical and categorical features (splitted in pre processing)
numeric_features = num_feat
categorical_features = cat_feat

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [3]:
valid_transmissions = ["MANUAL", "AUTOMATIC", "SEMIAUTO"]
valid_fueltypes    = ["PETROL", "DIESEL", "HYBRID"]

# KFold config
N_SPLITS = 8
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

lasso_param_distributions = {
    "alpha": np.logspace(-3, 1, 40)  # 1e-3 to 10
}

N_RANDOM_CONFIGS_LASSO = 40

lasso_sampler = ParameterSampler(
    lasso_param_distributions,
    n_iter=N_RANDOM_CONFIGS_LASSO,
    random_state=RANDOM_STATE
)

lasso_search_results = []

lasso_best_rmse = np.inf
lasso_best_config_rmse = None

lasso_best_mae = np.inf
lasso_best_config_mae = None

lasso_best_combo = np.inf
lasso_best_config_combo = None

lasso_log_path = "lasso_random_search_log.txt"

with open(lasso_log_path, "w", encoding="utf-8") as log_file:

    def log_lasso(msg: str):
        log_file.write(datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ") + msg + "\n")
        log_file.flush()

    log_lasso("# =============================")
    log_lasso("# START OF RANDOM SEARCH Lasso")
    log_lasso("# =============================")
    log_lasso(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS_LASSO}")
    log_lasso(f"param_distributions = {lasso_param_distributions}")

    for config_id, params in enumerate(lasso_sampler, start=1):
        log_lasso("")
        log_lasso(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS_LASSO} ########")
        log_lasso(f"Parameters: {params}")

        fold_rmses_val = []
        fold_maes_val  = []
        fold_r2s_val   = []
        fold_bias_val  = []

        fold_rmses_tr = []
        fold_maes_tr  = []
        fold_r2s_tr   = []
        fold_bias_tr  = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            log_lasso("")
            log_lasso(f"[CONFIG {config_id}] ==== FOLD {fold}/{N_SPLITS} ====")

            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log_lasso(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log_lasso(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # numeric imputers
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            tax_state = fit_tax_imputer(X_train, tax_col="tax", do_abs=True)
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            owners_state = fit_previous_owners_imputer(
                X_train, owners_col="previousOwners", year_col="year", mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # resolvers
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train, valid_brands=valid_brands,
                brand_col="Brand", model_col="model", year_col="year"
            )
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

            model_state = fit_invalid_model_resolver(
                train_df=X_train, valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand", model_col="model", year_col="year",
                fuel_col="fuelType", mpg_col="mpg"
            )
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val,   _, _ = transform_invalid_models(X_val,   model_state)

            transm_state = fit_transmission_resolver(
                train_df=X_train, valid_transmissions=valid_transmissions,
                transm_col="transmission", brand_col="Brand",
                model_col="model", fuel_col="fuelType"
            )
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

            fuel_state = fit_fueltype_resolver(
                train_df=X_train, valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType", brand_col="Brand",
                model_col="model", transm_col="transmission"
            )
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

            # encoding
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # scaling
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df,   X_val_cat],   axis=1)

            # model
            lasso = Lasso(
                alpha=params["alpha"],
                fit_intercept=True,
                max_iter=10000
            )

            log_lasso(f"[C{config_id}|F{fold}] Training Lasso...")
            lasso.fit(X_train_final, y_train)

            y_pred_train = lasso.predict(X_train_final)
            y_pred_val   = lasso.predict(X_val_final)

            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)
            bias_val = float(np.mean(y_pred_val - y_val))

            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)
            bias_tr = float(np.mean(y_pred_train - y_train))

            log_lasso(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.2f} | MAE: {mae_tr:.2f} | R2: {r2_tr:.4f} | Bias(pred-true): {bias_tr:.2f}")
            log_lasso(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.2f} | MAE: {mae_val:.2f} | R2: {r2_val:.4f} | Bias(pred-true): {bias_val:.2f}")

            fold_rmses_tr.append(rmse_tr)
            fold_maes_tr.append(mae_tr)
            fold_r2s_tr.append(r2_tr)
            fold_bias_tr.append(bias_tr)

            fold_rmses_val.append(rmse_val)
            fold_maes_val.append(mae_val)
            fold_r2s_val.append(r2_val)
            fold_bias_val.append(bias_val)

        # mean over folds
        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)

        mean_rmse_tr = np.mean(fold_rmses_tr)
        mean_mae_tr  = np.mean(fold_maes_tr)
        mean_r2_tr   = np.mean(fold_r2s_tr)
        mean_bias_tr = np.mean(fold_bias_tr)

        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log_lasso(f"Config {config_id} - Avg TRAIN RMSE: {mean_rmse_tr:.2f} | MAE: {mean_mae_tr:.2f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.2f}")
        log_lasso(f"Config {config_id} - Avg VAL   RMSE: {mean_rmse_val:.2f} | MAE: {mean_mae_val:.2f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.2f}")
        log_lasso(f"Config {config_id} - Combined score (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.2f}")

        lasso_search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "bias_train_mean": mean_bias_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "bias_mean": mean_bias_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < lasso_best_rmse:
            lasso_best_rmse = mean_rmse_val
            lasso_best_config_rmse = {**params}
            log_lasso(f"[NEW BEST RMSE] Config {config_id} with avg RMSE (VAL) = {lasso_best_rmse:.2f}")

        if mean_mae_val < lasso_best_mae:
            lasso_best_mae = mean_mae_val
            lasso_best_config_mae = {**params}
            log_lasso(f"[NEW BEST MAE] Config {config_id} with avg MAE (VAL) = {lasso_best_mae:.2f}")

        if combo_score < lasso_best_combo:
            lasso_best_combo = combo_score
            lasso_best_config_combo = {**params}
            log_lasso(f"[NEW BEST COMBINED] Config {config_id} with score = {lasso_best_combo:.2f}")

    log_lasso("")
    log_lasso("# =============================")
    log_lasso("# END OF RANDOM SEARCH Lasso")
    log_lasso("# =============================")
    log_lasso(f"Best configuration (min RMSE VAL): {lasso_best_config_rmse}")
    log_lasso(f"Best average RMSE (VAL): {lasso_best_rmse:.2f}")
    log_lasso(f"Best configuration (min MAE VAL): {lasso_best_config_mae}")
    log_lasso(f"Best average MAE  (VAL): {lasso_best_mae:.2f}")
    log_lasso(f"Best configuration (combined score VAL): {lasso_best_config_combo}")
    log_lasso(f"Best combined score (VAL): {lasso_best_combo:.2f}")

lasso_results_df = pd.DataFrame(lasso_search_results)
lasso_results_df_sorted = lasso_results_df.sort_values(by="mae_mean", ascending=True)

display(lasso_results_df_sorted.head(10))

print("\nBest configuration found (min RMSE VAL):")
print(lasso_best_config_rmse)
print("Best average RMSE (VAL):", lasso_best_rmse)

print("\nBest configuration found (min MAE VAL):")
print(lasso_best_config_mae)
print("Best average MAE (VAL):", lasso_best_mae)

print("\nBest configuration found (VAL = 0.5*RMSE + 0.5*MAE):")
print(lasso_best_config_combo)
print("Best combined score (VAL):", lasso_best_combo)

lasso_results_df_sorted.to_csv("lasso_random_search_results.csv", index=False)
print(f"\nDetailed logs in: {lasso_log_path}")

/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.348e+11, tolerance: 6.343e+08
  model = cd_fast.enet_coordinate_descent(
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.341e+11, tolerance: 6.312e+08
  model = cd_fast.enet_coordinate_descent(
/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing reg

,config_id,alpha,rmse_train_mean,mae_train_mean,r2_train_mean,bias_train_mean,rmse_mean,mae_mean,r2_mean,bias_mean,combo_score
39,40,10.000000,4159.391697,2643.581245,0.817515,-8.728905e-13,4184.834800,2649.830716,0.815216,0.139628,3417.332758
38,39,7.896523,4159.111854,2644.333522,0.817539,-8.299777e-13,4184.513273,2650.561357,0.815245,0.141076,3417.537315
37,38,6.235507,4158.937344,2644.980425,0.817555,-1.305491e-12,4184.305546,2651.196722,0.815263,0.142182,3417.751134
36,37,4.923883,4158.828525,2645.520267,0.817564,-8.020248e-13,4184.170418,2651.733159,0.815275,0.143059,3417.951788
35,36,3.888155,4158.760669,2645.962493,0.817570,-2.329967e-13,4184.081648,2652.172586,0.815283,0.143740,3418.127117
34,35,3.070291,4158.638063,2646.244583,0.817581,-6.325265e-13,4184.004062,2652.513753,0.815290,0.128935,3418.258908
33,34,2.424462,4158.444347,2646.384146,0.817598,-6.850566e-13,4183.847679,2652.690558,0.815304,0.126073,3418.269118
32,33,1.914482,4158.298910,2646.516478,0.817611,-5.382975e-13,4183.699695,2652.820877,0.815317,0.129063,3418.260286
31,32,1.511775,4158.202962,2646.657691,0.817619,-1.477494e-12,4183.592049,2652.932657,0.815327,0.121594,3418.262353
30,31,1.193777,4158.146094,2646.791375,0.817624,-4.532700e-13,4183.528462,2653.048618,0.815332,0.122145,3418.288540



Best configuration found (min RMSE VAL):
{'alpha': np.float64(0.03455107294592218)}
Best average RMSE (VAL): 4183.405022647017

Best configuration found (min MAE VAL):
{'alpha': np.float64(10.0)}
Best average MAE (VAL): 2649.830716428547

Best configuration found (VAL = 0.5*RMSE + 0.5*MAE):
{'alpha': np.float64(10.0)}
Best combined score (VAL): 3417.3327582326806

Detailed logs in: lasso_random_search_log.txt


In [4]:
# Choose final hyperparameters 
final_config_lasso = lasso_best_config_rmse
#final_config_lasso = {
#    "alpha": lasso_best_config_rmse["alpha"]
#}
print("Final Lasso config used for train:", final_config_lasso)

print("Preparing final Lasso Regression training...")

# 1) PREPARE FULL TRAIN FEATURES
X_full = X.copy()
y_full = y.copy()

# a) STRING NORMALIZATION
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = X_full[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

# Feature sets
high_card_features = ["Brand", "model"] 
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# 2) NUMERIC PRE PROCESSING - FULL TRAIN
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
# X_full = transform_paint_quality_imputer(X_full, state=paint_state)

owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

# 3) CATEGORICAL RESOLVERS - FULL TRAIN
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full, valid_brands=valid_brands, 
    brand_col="Brand", model_col="model", year_col="year"
)
X_full, _, brand_still_invalid_full = transform_ambiguous_brands(X_full, brand_state)
print(f"Train full - Invalid Brands remaining: {len(brand_still_invalid_full)}")

model_state = fit_invalid_model_resolver(
    train_df=X_full, valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand", model_col="model", year_col="year", 
    fuel_col="fuelType", mpg_col="mpg"
)
X_full, _, model_still_invalid_full = transform_invalid_models(X_full, model_state)
print(f"Train full - Invalid Models remaining: {len(model_still_invalid_full)}")

transm_state = fit_transmission_resolver(
    train_df=X_full, valid_transmissions=valid_transmissions,
    transm_col="transmission", brand_col="Brand", 
    model_col="model", fuel_col="fuelType"
)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(
    train_df=X_full, valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType", brand_col="Brand", 
    model_col="model", transm_col="transmission"
)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# 4) CATEGORICAL ENCODING - FULL TRAIN
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)
print("X_full_cat shape:", X_full_cat.shape)

# 5) NUMERIC SCALING - FULL TRAIN
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)
print("X_full_num_df shape:", X_full_num_df.shape)

# 6) FINAL MATRIX - FULL TRAIN
X_full_final = pd.concat([X_full_num_df, X_full_cat], axis=1)
print("X_full_final shape:", X_full_final.shape)

# 7) TRAIN FINAL LASSO REGRESSION MODEL
lasso_final = Lasso(
    alpha=final_config_lasso["alpha"],
    fit_intercept=True,
    max_iter=10000
)

print("Training final Lasso Regression model on full data...")
lasso_final.fit(X_full_final, y_full)
print("Done.")

# 8) PREPARE TEST FEATURES
test_df = pd.read_csv("../../project_data/test.csv")

for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

X_test = test_df[numeric_features + categorical_features].copy()

# NUMERIC PREPROCESSING - TEST
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
# X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# CATEGORICAL RESOLVERS - TEST
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# ENCODING - TEST
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# SCALING - TEST
X_test_num = scaler.transform(X_test[numeric_features])
X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# FINAL MATRIX & PREDICTION
X_test_final = pd.concat([X_test_num_df, X_test_cat], axis=1)
X_test_final = X_test_final[X_full_final.columns]

print("X_test_final shape:", X_test_final.shape)

y_test_pred = lasso_final.predict(X_test_final)

print("Predictions summary (float):")
print(pd.Series(y_test_pred).describe())

y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred  
})

sub_name = f"lasso_regression_final_submission_alpha{final_config_lasso['alpha']:.4f}.csv"
submission.to_csv(sub_name, index=False)
print(f"Submission file created: {sub_name}")

Final Lasso config used for train: {'alpha': np.float64(0.03455107294592218)}
Preparing final Lasso Regression training...
Train full - Invalid Brands remaining: 0
Train full - Invalid Models remaining: 1
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 6)
X_full_final shape: (75973, 16)
Training final Lasso Regression model on full data...


/opt/anaconda3/envs/Fall2526/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.664e+10, tolerance: 7.203e+08
  model = cd_fast.enet_coordinate_descent(


Done.
X_test_final shape: (32567, 16)
Predictions summary (float):
count    32567.000000
mean     16897.729057
std       8754.387309
min     -19423.046246
25%      10847.795719
50%      15466.501498
75%      22288.365654
max      95024.693605
dtype: float64
Submission file created: lasso_regression_final_submission_alpha0.0346.csv
